# Lane 3: Energy-Search - Google Colab Runner
**Project:** Lyapunov Residual Stream Pruning for Reasoning  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Can model-intrinsic dynamical instability metrics detect reasoning derailment and prune bad rollout branches without an external PRM?

Optimized for Google Colab T4 and local GPUs with KV-caching.


In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time, math
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    print(f"Device: {dev_name}")
    torch.set_float32_matmul_precision("high")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'energy-search'
    if not os.path.exists(repo_dir):
        print('Cloning energy-search (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/energy-search.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication (with 90s Idle-Kill Guard)
# Securely reads from Colab Secrets (Key icon on left sidebar) or reuses ~/.netrc.
# If interactive input hangs for > 90 seconds, automatically unassigns GPU to save credits.
import os
import signal
import sys
import wandb

def auth_timeout_handler(signum, frame):
    print("\n[Credit Guard Alert] W&B authentication timed out (90s elapsed without input)!")
    print("[Credit Guard Alert] Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
    raise TimeoutError("Interactive authentication timed out after 90 seconds to protect credits.")

if hasattr(signal, 'SIGALRM'):
    signal.signal(signal.SIGALRM, auth_timeout_handler)
    signal.alarm(90)

try:
    # 1. Attempt reading from Google Colab Secrets across candidate names
    try:
        from google.colab import userdata
        for secret_name in ['WANDB_API_KEY', 'wandb_api_key', 'WANDB', 'WAN', 'wandb']:
            try:
                val = userdata.get(secret_name)
                if val and len(val.strip()) > 10:
                    os.environ['WANDB_API_KEY'] = val.strip()
                    print(f"[W&B] Loaded API key from Colab Secrets ('{secret_name}').")
                    break
            except Exception:
                pass
    except Exception:
        pass

    # 2. Fallback: check ~/.netrc if previously logged in
    netrc_path = os.path.expanduser('~/.netrc')
    if 'WANDB_API_KEY' not in os.environ and os.path.exists(netrc_path):
        try:
            with open(netrc_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2 and parts[0] == 'password' and len(parts[1]) > 10:
                        os.environ['WANDB_API_KEY'] = parts[1]
                        print("[W&B] Restored WANDB_API_KEY automatically from ~/.netrc.")
                        break
        except Exception:
            pass

    # 3. Login to W&B without prompting if credentials exist
    if 'WANDB_API_KEY' in os.environ:
        wandb.login(key=os.environ['WANDB_API_KEY'], relogin=False)
    else:
        print("[W&B] Note: To avoid interactive prompt, name your secret 'WANDB_API_KEY' and paste full key into Value.")
        wandb.login(relogin=False)

    os.environ['WANDB_ENTITY'] = 'research-projekts'
    print('[W&B] Authenticated successfully. Target entity: research-projekts')
finally:
    if hasattr(signal, 'SIGALRM'):
        signal.alarm(0)


In [ ]:
# Step 2: Dependencies & Standalone Engine (Dual-mode)
try:
    from energy_search.sampler import sample_with_energy_pruning
    from energy_search.pruner import DynamicalInstabilityPruner
    from energy_search.telemetry import compute_relative_velocity, compute_token_entropy
    print("Loaded Energy-Search from installed package.")
except ImportError:
    print("Package not found on path; loading self-contained standalone implementation...")
    import math
    from typing import List, Tuple, Dict, Any, Optional
    
    def compute_relative_velocity(h_curr, h_prev, eps=1e-6):
        with torch.inference_mode():
            diff = torch.norm(h_curr.float() - h_prev.float(), p=2)
            norm_c = torch.norm(h_curr.float(), p=2)
            return (diff / (norm_c + eps)).item()
    
    def compute_token_entropy(logits, eps=1e-9):
        with torch.inference_mode():
            probs = torch.softmax(logits.float(), dim=-1)
            return -torch.sum(probs * torch.log(probs + eps)).item()
    
    class RollingVelocityStats:
        def __init__(self, window_size=16):
            self.window_size = window_size
            self.history: List[float] = []
        def update_and_normalize(self, v_rel, eps=1e-6):
            self.history.append(v_rel)
            if len(self.history) < 2: return 0.0
            win = self.history[-self.window_size:]
            mu = sum(win) / len(win)
            var = sum((x - mu)**2 for x in win) / max(1, len(win) - 1)
            return max(0.0, (v_rel - mu) / (math.sqrt(var) + eps))
    
    class DynamicalInstabilityPruner:
        def __init__(self, tau=2.5, k_persist=4, freeze_vel_tau=0.05, freeze_ent_tau=0.15, k_freeze=6, window_size=16):
            self.tau, self.k_persist = tau, k_persist
            self.freeze_vel_tau, self.freeze_ent_tau, self.k_freeze = freeze_vel_tau, freeze_ent_tau, k_freeze
            self.stats = RollingVelocityStats(window_size=window_size)
            self.consecutive_spikes = 0
            self.consecutive_freezes = 0
            self.prompt_entropy = 1.0
        def set_prompt_entropy(self, ent): self.prompt_entropy = max(0.1, ent)
        def evaluate_step(self, v_rel, ent) -> Tuple[bool, float, str]:
            z = self.stats.update_and_normalize(v_rel)
            inst = z * (ent / self.prompt_entropy)
            if inst > self.tau: self.consecutive_spikes += 1
            else: self.consecutive_spikes = 0
            if v_rel < self.freeze_vel_tau and ent < self.freeze_ent_tau: self.consecutive_freezes += 1
            else: self.consecutive_freezes = 0
            if self.consecutive_spikes >= self.k_persist: return True, inst, 'chaotic_divergence'
            if self.consecutive_freezes >= self.k_freeze: return True, inst, 'cognitive_freeze_loop'
            return False, inst, 'active'
    
    def sample_with_energy_pruning(model, input_ids, max_new_tokens=40, tau=2.5, k_persist=4, freeze_vel_tau=0.05, freeze_ent_tau=0.15, k_freeze=6, device=None):
        pruner = DynamicalInstabilityPruner(tau=tau, k_persist=k_persist, freeze_vel_tau=freeze_vel_tau, freeze_ent_tau=freeze_ent_tau, k_freeze=k_freeze)
        gen = input_ids.clone()
        h_hist = []
        inst_hist = []
        pruned_step = None
        reason = 'completed'
        with torch.inference_mode():
            for s in range(max_new_tokens):
                out = model(gen)
                logits, h_all = out if isinstance(out, tuple) else (out, out)
                h_last = h_all[:, -1, :]
                logits_last = logits[:, -1, :]
                ent = compute_token_entropy(logits_last)
                v_rel = 0.0 if s == 0 else compute_relative_velocity(h_last, h_hist[-1])
                if s == 0: pruner.set_prompt_entropy(ent)
                prune, inst, r = pruner.evaluate_step(v_rel, ent)
                h_hist.append(h_last)
                inst_hist.append(inst)
                if prune:
                    pruned_step = s
                    reason = r
                    break
                nxt = torch.argmax(logits_last, dim=-1, keepdim=True)
                gen = torch.cat([gen, nxt], dim=-1)
        return {'tokens_generated': len(h_hist), 'pruned': pruned_step is not None, 'pruned_at_step': pruned_step, 'prune_reason': reason, 'mean_instability': sum(inst_hist)/max(1, len(inst_hist))}
    print("Standalone Energy-Search initialized.")


In [ ]:
# Step 3: Run KV-Cached Pruning Smoke Test
import torch.nn as nn

class ToyTransformerLM(nn.Module):
    def __init__(self, vocab_size=100, d_model=64, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=128, batch_first=True)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        h = self.embed(token_ids)
        for layer in self.layers:
            h = layer(h)
        logits = self.head(h)
        return logits, h

model = ToyTransformerLM().to(device)
model.eval()

prompt = torch.randint(0, 100, (1, 10), device=device)
t0 = time.time()
res = sample_with_energy_pruning(
    model=model,
    input_ids=prompt,
    max_new_tokens=40,
    tau=2.5,
    k_persist=4,
    freeze_vel_tau=0.05,
    freeze_ent_tau=0.15,
    k_freeze=6,
    device=device
)
elapsed = time.time() - t0
latency_ms = (elapsed / max(1, res['tokens_generated'])) * 1000.0

t_tel0 = time.time()
n_tel = 200
dummy_logits = torch.randn(1, 100, device=device)
dummy_h1 = torch.randn(1, 64, device=device)
dummy_h2 = torch.randn(1, 64, device=device)
for _ in range(n_tel):
    _ = compute_token_entropy(dummy_logits)
    _ = compute_relative_velocity(dummy_h1, dummy_h2)
telemetry_ms = ((time.time() - t_tel0) / n_tel) * 1000.0

print(f"Tokens Generated: {res['tokens_generated']} | Time: {elapsed:.3f}s")
print(f"End-to-End Generation Latency: {latency_ms:.2f} ms/token")
print(f"Pure Telemetry Overhead: {telemetry_ms:.3f} ms/token (Target: < 5.0 ms)")
print(f"Pruned: {res['pruned']} (Step: {res['pruned_at_step']}, Reason: {res['prune_reason']})")
print(f"Mean Instability Index: {res['mean_instability']:.4f}")


In [ ]:
# Step 4: Pre-Registered Gate Assessment
passed = telemetry_ms < 5.0
print("=" * 60)
print(f"Telemetry Overhead: {telemetry_ms:.3f} ms/token (Target: < 5.0 ms)")
print(f"Pre-Registered Gate: {'PASSED (Low overhead telemetry)' if passed else 'FAILED (Kill)'}")
print("=" * 60)


In [ ]:
# Step 4b: Stage 2 GSM8K Reasoning Benchmark with Instability Pruning (Requires A100 GPU)
!pip install -q transformers accelerate datasets
!python scripts/eval_gsm8k.py --model-name Qwen/Qwen2.5-Math-7B-Instruct --num-samples 150 --tau 2.5 --k-persist 4 --wandb --wandb-project research-projekts --wandb-name energy-search-qwen7b-a100


In [ ]:
# Step Pre-Kill: Explicit W&B Telemetry Cloud Upload
# Ensures every run (online or offline) is completely synced and uploaded to W&B cloud before teardown
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true
!wandb sync --include-offline /tmp/wandb wandb 2>/dev/null || true


In [ ]:
# Step Final: Credit Protection & Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
# Set to False by default so the virtual machine stays alive after execution
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
else:
    print("\n[Credit Guard] AUTO_KILL_ON_FINISH is False. Keeping VM instance alive for inspection.")
